In [3]:
import sys
sys.path.append("../src")

In [4]:
import numpy as np
import pandas as pd

from bb84 import run_bb84_with_eve
from error_correction import parity_error_correction
from encryption import run_secure_message_exchange

In [6]:
bb84_result = run_bb84_with_eve(
    n=10000,
    eve_intercept_prob=0.05
)

alice_key = bb84_result["alice_key"]
bob_key = bb84_result["bob_key"]

print("QBER: ", bb84_result["qber"])
print("Raw key length: ", len(alice_key))
print("Raw mismatches: ", np.sum(alice_key != bob_key))

QBER:  0.012502520669489817
Raw key length:  4959
Raw mismatches:  62


In [7]:
correction_result = parity_error_correction(
    alice_key,
    bob_key,
    block_size=16,
    passes=5
)

print("Corrections applied:", correction_result["corrections_applied"])
print("Parity checks:", correction_result["parity_checks"])
print("Final mismatches:", correction_result["final_mismatches"])
print("Success:", correction_result["success"])

Corrections applied: 62
Parity checks: 1798
Final mismatches: 0
Success: True


In [8]:
message_result = run_secure_message_exchange(
    message="Hello quantum world",
    n_qubits=10000,
    eve_intercept_prob=0.05,
    qber_threshold=0.11,
    use_error_correction=True,
    error_correction_block_size=16,
    error_correction_passes=5
)

print("Status:", message_result["status"])
print("Reason:", message_result["reason"])
print("QBER:", message_result["qber"])
print("Raw mismatches:", message_result["raw_mismatches"])
print("Final mismatches:", message_result["final_mismatches"])
print("Corrections applied:", message_result["corrections_applied"])
print("Original message:", message_result["message"])
print("Decrypted message:", message_result["decrypted_message"])

Status: success
Reason: Message encrypted and decrypted successfully.
QBER: 0.01277445109780439
Raw mismatches: 64
Final mismatches: 0
Corrections applied: 64
Original message: Hello quantum world
Decrypted message: Hello quantum world


## Error Correction

This notebook demonstrates a simplified parity-based error-correction step for BB84.

After Alice and Bob generate sifted keys, their keys may still contain mismatches if Eve's interception probability is low enough to stay below the QBER threshold. Real BB84 systems use reconciliation protocols to correct these errors.

This version uses a parity-based method:
1. Split the key into blocks.
2. Compare Alice and Bob's block parities.
3. If parity differs, use binary parity checks to locate one likely error.
4. Flip Bob's bit.
5. Repeat across several shuffled passes.